In [ ]:
import pickle  
import networkx as nx
import random
import numpy as np
import copy
import pandas as pd
import itertools
import math
from tqdm import tqdm
from scipy import integrate 
from collections import defaultdict
from find_motifs import * 
from Motif_structures_1216 import *
# import network_embedding_new as ne
from similarity_nodeijs import *

moti = [globals()[f'M{i}'] for i in range(1, 546)]
moti_name = [f'M{i}' for i in range(1, 546)]

motif_index = []
zip1 = zip(moti, moti_name)
for i, j in zip1:
    motif_index.append((i, j))

noti = [globals()[f'N{i}'] for i in range(1, 481)]
noti_name = [f'N{i}' for i in range(1, 481)]
notif_index = []
zip2 = zip(noti, noti_name)
for i, j in zip2:
    notif_index.append((i, j))

In [ ]:
import pickle
import pandas as pd
import networkx as nx
import random
import copy
import numpy as np
from collections import defaultdict
from tqdm import tqdm
from pathlib import Path
import datetime

pickle_path = Path("D:/Orbit_degree_LP/OLP_updated.pickle")
save_dir = Path("D:/Orbit_degree_LP/jia/all_3_6/1216")
save_dir.mkdir(parents=True, exist_ok=True)
log_file = save_dir / "completed_log.txt"

with open(pickle_path, "rb") as infile:
    df = pickle.load(infile)

df_edgelists = df["edges_id"]
all_name = list(df["network_name"])

pending_indices = [
    i for i in range(len(df))
    if not (save_dir / f"n_{i}_features.csv").exists()
]

print(f"共 {len(df)} 个网络，已有 {len(df) - len(pending_indices)} 个完成，"
      f"剩余 {len(pending_indices)} 个待计算。\n")

for i in tqdm(pending_indices, total=len(pending_indices), desc="未完成网络进度"):
    try:
        file_name = save_dir / f"n_{i}_features.csv"

        edges_orig = df_edgelists.iloc[i]
        file = df["network_name"][i]
        num_edges = df["number_edges"][i]
        num_nodes = df["number_nodes"][i]
        ave_degree = df["ave_degree"][i]
        domain = df["networkDomain"][i]

        g1 = nx.Graph()
        g1.add_edges_from(edges_orig)

        nodes = list(g1.nodes())
        all_edges = list(g1.edges())
        remove_edge_num = int(len(all_edges) * 0.2)

        train_g = copy.deepcopy(g1)
        remove_edges = random.sample(all_edges, remove_edge_num)
        train_g.remove_edges_from(remove_edges)
        positive_samples = remove_edges

        negitive_samples = []
        while len(negitive_samples) < len(positive_samples):
            u = random.choice(nodes)
            v = random.choice(nodes)
            if (
                u != v
                and (u, v) not in all_edges
                and (v, u) not in all_edges
                and (u, v) not in negitive_samples
                and (v, u) not in negitive_samples
            ):
                negitive_samples.append((u, v))

        X = defaultdict(list)
        X["edge"] = positive_samples + negitive_samples
        X["label"] = [1] * len(positive_samples) + [0] * len(negitive_samples)
        all_samples = positive_samples + negitive_samples

        individual_nodes = sorted(set(element for tup in all_samples for element in tup))
        node_orbit_degree_dict = {}
        for node in individual_nodes:
            node_orbit_degree = []
            for n in noti:
                node_orbit_degree.append(node_orbit_motif_degree(train_g, n, node, 1))
            node_orbit_degree_dict[node] = node_orbit_degree

        for j in range(len(noti_name)):
            res = []
            for one_node_pair in all_samples:
                res.append(
                    node_orbit_degree_dict[one_node_pair[0]][j]
                    * node_orbit_degree_dict[one_node_pair[1]][j]
                )
            X[noti_name[j]] = res

        for m in motif_index:
            res = []
            for one in all_samples:
                a = edge_orbit_motif_degree(train_g, m[0], one, (1, 2))
                res.append(a)
            X[m[1]] = res

        for one_node in nodes:
            if not train_g.has_node(one_node):
                train_g.add_node(one_node)

        for edge in train_g.edges():
            train_g[edge[0]][edge[1]]["weight"] = 1

        df_features = pd.DataFrame(X)
        df_features.to_csv(file_name, index=False)

        with open(log_file, "a", encoding="utf-8") as log:
            log.write(f"{datetime.datetime.now()} | 完成网络 {i}: {file_name.name}\n")

    except Exception as e:
        print(f"第 {i} 个网络出错: {e}")
        continue

print("全部任务完成。日志已保存至：", log_file)

In [ ]:
import pickle
import pandas as pd
import networkx as nx
from pathlib import Path
from tqdm import tqdm

pickle_path = Path("D:/Orbit_degree_LP/OLP_updated.pickle")
save_dir = Path("D:/Orbit_degree_LP/jia/all_3_6")
save_dir.mkdir(parents=True, exist_ok=True)
save_file = save_dir / "network_basic_info.csv"

with open(pickle_path, "rb") as infile:
    df = pickle.load(infile)

edges_all = df["edges_id"]

records = []

for i in tqdm(range(len(edges_all)), desc="计算网络指标中"):
    try:
        edges = edges_all.iloc[i]
        G = nx.Graph()
        G.add_edges_from(edges)

        num_nodes = G.number_of_nodes()
        num_edges = G.number_of_edges()
        ave_degree = 2 * num_edges / num_nodes if num_nodes > 0 else 0
        ave_clustering = nx.average_clustering(G) if num_nodes > 0 else 0

        records.append({
            "net": i,
            "num_nodes": num_nodes,
            "ave_degree": round(ave_degree, 4),
            "ave_clustering": round(ave_clustering, 4)
        })
    except Exception as e:
        print(f"网络 {i} 出错: {e}")
        records.append({
            "net": i,
            "num_nodes": None,
            "ave_degree": None,
            "ave_clustering": None
        })
        continue

result_df = pd.DataFrame(records)
result_df.to_csv(save_file, index=False, encoding="utf-8-sig")
print(f"所有550个网络指标已计算完成，保存至：{save_file}")

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

def draw_square_tiling(m=10, n=10):
    G = nx.grid_2d_graph(m, n)
    pos = {(i, j): (j, -i) for i, j in G.nodes()} 

    plt.figure(figsize=(6, 6))
    nx.draw(
        G, pos,
        with_labels=False,
        node_size=50,
        width=0.8
    )
    plt.axis('equal')
    plt.axis('off')
    plt.tight_layout()
    plt.show()
    return G

G = draw_square_tiling(10, 10)

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm import tqdm
from collections import defaultdict
import math
import itertools
from scipy import integrate
import random
from find_motifs import *       
from Motif_structures_1216 import *
from similarity_nodeijs import *

OUTPUT_CSV = r"D:\Orbit_degree_LP\jia\all_3_6\square_grid_10x10_link_samples_100pairs_features.csv"

m, n = 10, 10

G_grid = nx.grid_2d_graph(m, n)

mapping = {node: idx for idx, node in enumerate(sorted(G_grid.nodes()))}
G = nx.relabel_nodes(G_grid, mapping)

print("节点数 =", G.number_of_nodes())
print("边数   =", G.number_of_edges())

moti = [globals()[f'M{i}'] for i in range(1, 546)]
moti_name = [f'M{i}' for i in range(1, 546)]
motif_index = []
for m_struct, m_name in zip(moti, moti_name):
    motif_index.append((m_struct, m_name))

# N1..N15
noti = [globals()[f'N{i}'] for i in range(1, 481)]
noti_name = [f'N{i}' for i in range(1, 481)]

notif_index = []
for n_struct, n_name in zip(noti, noti_name):
    notif_index.append((n_struct, n_name))

edges = list(G.edges())
nodes = list(G.nodes())

num_pos = 50
if len(edges) <= num_pos:
    positive_samples = edges[:]  # 边不够就全用
else:
    positive_samples = random.sample(edges, num_pos)

num_neg = 50
negative_samples = []

edge_set = set(edges) | { (v, u) for (u, v) in edges }  # 无向边两种顺序都排除

while len(negative_samples) < num_neg:
    u = random.choice(nodes)
    v = random.choice(nodes)
    if u == v:
        continue
    if (u, v) in edge_set or (v, u) in edge_set:
        continue
    if (u, v) in negative_samples or (v, u) in negative_samples:
        continue
    negative_samples.append((u, v))

print(f"正样本（存在边）数量: {len(positive_samples)}")
print(f"负样本（不存在边）数量: {len(negative_samples)}")

# 合并成总样本
all_samples = positive_samples + negative_samples
labels = [1] * len(positive_samples) + [0] * len(negative_samples)

sampled_nodes = sorted(set(u for e in all_samples for u in e))

print("\n开始计算节点轨道度（N*，按节点预计算）...")

node_orbit_degree_dict = {} 

for node in tqdm(sampled_nodes, desc="节点循环 (N* 单点轨道度)"):
    node_orbits = []
    for n_struct, n_name in notif_index:
        val = node_orbit_motif_degree(G, n_struct, node, 1)
        node_orbits.append(val)
    node_orbit_degree_dict[node] = node_orbits

print("节点轨道度预计算完成。")

print("\n开始按节点对构造特征表（每对节点一行）...")

X = defaultdict(list)

for (u, v), lab in zip(all_samples, labels):
    X["u"].append(u)
    X["v"].append(v)
    X["label"].append(lab)

for j, n_name in enumerate(noti_name):
    vals = []
    for (u, v) in all_samples:
        val_u = node_orbit_degree_dict[u][j]
        val_v = node_orbit_degree_dict[v][j]
        vals.append(val_u * val_v)
    X[n_name] = vals

print("\n开始计算每对节点的 M* 模体特征...")

for m_struct, m_name in tqdm(motif_index, desc="模体类型 (M*)"):
    vals = []
    for e in all_samples:
        val = edge_orbit_motif_degree(G, m_struct, e, (1, 2))
        vals.append(val)
    X[m_name] = vals

print("节点对级 M* 特征计算完成。")

df_features = pd.DataFrame(X)
df_features.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

print(f"共 {len(all_samples)} 对节点的 N* + M* 特征已保存到：{OUTPUT_CSV}")